> **Public snapshot note**
> 이 노트북은 원래 연구 실행 흐름을 보존한 archival artifact입니다.
> 공개 스냅샷에서는 raw PDF, processed corpus, `kg_gen/graphrag_workspace/input`, `kg_gen/graphrag_workspace/output`,
> `kg_gen/snu_kg_output/graphrag_workspace/output*` 등 bulk/source-derived 자산을 의도적으로 제외했습니다.
> 따라서 아래의 일부 경로 설명은 **원래 실험 환경 기준**이며, 현재 public snapshot과 1:1로 대응하지 않을 수 있습니다.


# 농업 텍스트 데이터 기반 QA 데이터셋 생성 파이프라인

이 노트북은 두 가지 주요 텍스트 추출 방식을 제공합니다:

## 📝 텍스트 추출 방식:
1. **📄 PDF 텍스트 추출**: arXiv 논문 PDF 파일에서 텍스트 추출  
2. **🌐 웹 스크래핑**: RDA URL 목록에서 농업 정보 추출

## 📁 파일 구조:
- `/sources/arxiv/`: arXiv 논문 PDF 파일들
- `/sources/rda_url.txt`: RDA 웹페이지 URL 목록 (한 줄에 하나씩)
- `/processed_texts/`: 처리된 텍스트 파일들
- `/qa_datasets/`: 생성된 QA 데이터셋들

## 1. 필요한 라이브러리 설치 및 임포트

In [ ]:
# 필요한 패키지 설치
# 기본 패키지
!pip install openai tqdm

# 웹 스크래핑용
!pip install requests beautifulsoup4

# PDF 처리용
!pip install PyPDF2 pdfplumber

# OCR용 (선택사항)
!pip install pytesseract pdf2image
# 주의: tesseract 바이너리도 별도 설치 필요 (conda install -c conda-forge tesseract)

In [ ]:
import os
import json
import time
import requests
from typing import List, Dict, Optional, Tuple
from datetime import datetime
from pathlib import Path
import logging
from tqdm import tqdm

# 웹 스크래핑
from bs4 import BeautifulSoup

# PDF 처리
import PyPDF2
import pdfplumber

# OpenAI API
from openai import OpenAI

# 로깅 설정
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

## 2. 환경 설정

In [ ]:
# OpenAI API 키 설정
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
if not OPENAI_API_KEY:
    OPENAI_API_KEY = input("OpenAI API 키를 입력하세요: ")

client = OpenAI(api_key=OPENAI_API_KEY)

# 경로 설정
BASE_DIR = Path("<PROJECT_ROOT>/rag_dataset")

# 소스 경로들
SOURCES_DIR = BASE_DIR / "sources"
ARXIV_DIR = SOURCES_DIR / "arxiv"          # arXiv 논문 PDF 파일들
RDA_URL_FILE = SOURCES_DIR / "rda_url.txt" # RDA URL 목록 파일

# 출력 폴더들
OUTPUT_DIR = BASE_DIR / "processed_texts"
QA_DIR = BASE_DIR / "qa_datasets"

# 디렉토리 생성
for dir_path in [ARXIV_DIR, OUTPUT_DIR, QA_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

# RDA URL 파일이 없으면 생성
if not RDA_URL_FILE.exists():
    RDA_URL_FILE.touch()
    print(f"📄 RDA URL 파일 생성됨: {RDA_URL_FILE}")

print("📁 파일 구조:")
print(f"   📂 arXiv PDF: {ARXIV_DIR}")
print(f"   📄 RDA URL 파일: {RDA_URL_FILE}")
print(f"   📂 처리된 텍스트: {OUTPUT_DIR}")
print(f"   📂 QA 데이터셋: {QA_DIR}")

## 3. 텍스트 추출 클래스들

### 3.1 웹 스크래핑 텍스트 추출기

In [ ]:
class WebScrapingExtractor:
    """
    웹 스크래핑을 통한 텍스트 추출 클래스
    이전 수집/전처리 노트북 방식을 기반으로 함
    """
    
    def __init__(self):
        self.extracted_count = 0
        self.failed_count = 0
        self.session = requests.Session()
        # User-Agent 설정
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        })
    
    def get_page_content(self, url: str, retries: int = 3) -> Optional[BeautifulSoup]:
        """웹페이지 내용 가져오기 (재시도 포함)"""
        for attempt in range(retries):
            try:
                response = self.session.get(url, timeout=10)
                if response.status_code == 200:
                    return BeautifulSoup(response.content, 'html.parser')
                elif response.status_code == 429:
                    retry_after = int(response.headers.get('Retry-After', 60))
                    print(f"⚠️  요청 제한. {retry_after}초 후 재시도...")
                    time.sleep(retry_after)
                else:
                    print(f"❌ HTTP {response.status_code}: {url}")
                    
            except Exception as e:
                print(f"❌ 시도 {attempt + 1} 실패: {str(e)[:100]}")
                if attempt < retries - 1:
                    time.sleep(2 ** attempt)  # 지수 백오프
                
        return None
    
    def extract_text_from_url(self, url: str) -> Tuple[str, Dict]:
        """URL에서 텍스트 추출"""
        metadata = {
            'url': url,
            'extracted_at': datetime.now().isoformat(),
            'method': '웹 스크래핑',
            'status': 'unknown'
        }
        
        print(f"🔍 {url} 웹 스크래핑 중...")
        
        soup = self.get_page_content(url)
        if not soup:
            self.failed_count += 1
            metadata['status'] = '페이지 로드 실패'
            return "", metadata
        
        # 다양한 텍스트 추출 방법 시도
        extraction_methods = [
            ('article 태그', lambda s: s.find_all('article')),
            ('main 태그', lambda s: s.find_all('main')),
            ('p 태그', lambda s: s.find_all('p')),
            ('div.content', lambda s: s.find_all('div', class_=['content', 'article-content', 'post-content'])),
            ('전체 body', lambda s: [s.find('body')] if s.find('body') else [])
        ]
        
        text_parts = []
        method_used = "unknown"
        
        for method_name, extract_func in extraction_methods:
            try:
                elements = extract_func(soup)
                if elements:
                    for element in elements:
                        if element:
                            text_parts.append(element.get_text(strip=True))
                    
                    # 의미있는 텍스트가 추출되었는지 확인
                    combined_text = ' '.join(text_parts)
                    if len(combined_text.strip()) > 200:
                        method_used = method_name
                        print(f"   ✅ {method_name}로 텍스트 추출: {len(combined_text):,}자")
                        break
                    else:
                        text_parts = []  # 재시도를 위해 초기화
            except Exception as e:
                print(f"   ❌ {method_name} 실패: {str(e)[:50]}")
                continue
        
        if text_parts:
            self.extracted_count += 1
            result_text = '\n\n'.join(text_parts)
            metadata['status'] = '성공'
            metadata['method_detail'] = method_used
            return result_text, metadata
        else:
            self.failed_count += 1
            metadata['status'] = '텍스트 추출 실패'
            return "", metadata
    
    def extract_from_url_list(self, urls: List[str], delay: float = 1.0) -> List[Dict]:
        """URL 목록에서 텍스트 추출"""
        results = []
        
        print(f"🌐 웹 스크래핑 시작: {len(urls)}개 URL")
        
        for i, url in enumerate(tqdm(urls, desc="웹 스크래핑")):
            text, metadata = self.extract_text_from_url(url)
            
            if text and len(text.strip()) > 100:
                results.append({
                    'text': text,
                    'metadata': metadata
                })
                print(f"✅ {i+1}/{len(urls)} 완료: {len(text):,}자 추출")
            else:
                print(f"❌ {i+1}/{len(urls)} 실패: {url}")
            
            # 서버 부하 방지를 위한 딜레이
            time.sleep(delay)
        
        print(f"\n📊 웹 스크래핑 완료: 성공 {self.extracted_count}개, 실패 {self.failed_count}개")
        return results
    
    def load_urls_from_file(self, file_path: Path) -> List[str]:
        """파일에서 URL 목록 로드"""
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                urls = [line.strip() for line in f if line.strip() and not line.startswith('#')]
            print(f"📋 URL 파일 로드: {len(urls)}개 URL ({file_path.name})")
            return urls
        except Exception as e:
            print(f"❌ URL 파일 로드 실패: {e}")
            return []

### 3.2 PDF 텍스트 추출기

In [ ]:
class PDFTextExtractor:
    """
    PDF 파일에서 텍스트를 추출하는 클래스
    한국어 PDF 처리에 최적화됨
    """
    
    def __init__(self):
        self.extracted_count = 0
        self.failed_count = 0
    
    def extract_text_from_pdf(self, pdf_path: Path) -> Tuple[str, Dict]:
        """
        PDF 파일에서 텍스트 추출 - 다양한 방법 시도
        """
        metadata = {
            'filename': pdf_path.name,
            'path': str(pdf_path),
            'size': pdf_path.stat().st_size,
            'extracted_at': datetime.now().isoformat(),
            'method': '알 수 없음',
            'page_count': 0
        }
        
        print(f"\n🔍 {pdf_path.name} PDF 텍스트 추출 중...")
        
        # 추출 방법들을 순서대로 시도
        extraction_methods = [
            ("pdfplumber (고급)", self._extract_with_pdfplumber_enhanced),
            ("pdfplumber (기본)", self._extract_with_pdfplumber_basic),
            ("PyPDF2", self._extract_with_pypdf2)
        ]
        
        for method_name, method_func in extraction_methods:
            try:
                print(f"   📝 {method_name} 시도 중...")
                text = method_func(pdf_path, metadata)
                
                # 텍스트 품질 검사
                if self._is_text_valid(text):
                    metadata['method'] = method_name
                    self.extracted_count += 1
                    print(f"   ✅ {method_name}로 성공! 텍스트 길이: {len(text):,}자")
                    return text, metadata
                else:
                    quality_score = self._get_text_quality_score(text)
                    print(f"   ❌ {method_name}: 텍스트 품질 불량 (길이: {len(text)}, 품질: {quality_score:.2f})")
                    
            except Exception as e:
                print(f"   ❌ {method_name} 실패: {str(e)[:100]}")
                continue
        
        # 모든 방법 실패
        print(f"   🚫 모든 PDF 추출 방법 실패")
        self.failed_count += 1
        metadata['error'] = "모든 PDF 추출 방법 실패"
        return "", metadata
    
    def _is_text_valid(self, text: str) -> bool:
        """텍스트가 유효한지 검사"""
        if not text or len(text.strip()) < 200:
            return False
        
        # 무의미한 문자 비율 검사
        meaningless_chars = ['(cid:', '\\x', '\\u', '<', '>']
        meaningless_count = sum(text.count(char) for char in meaningless_chars)
        meaningless_ratio = meaningless_count / len(text)
        
        # 한글이나 영문 비율 검사
        korean_count = sum(1 for char in text if '\uac00' <= char <= '\ud7a3')
        english_count = sum(1 for char in text if char.isalpha() and ord(char) < 128)
        meaningful_ratio = (korean_count + english_count) / len(text)
        
        return meaningless_ratio < 0.3 and meaningful_ratio > 0.1
    
    def _get_text_quality_score(self, text: str) -> float:
        """텍스트 품질 점수 계산 (0-1)"""
        if not text:
            return 0.0
        
        # 다양한 품질 지표들
        korean_ratio = sum(1 for char in text if '\uac00' <= char <= '\ud7a3') / len(text)
        english_ratio = sum(1 for char in text if char.isalpha() and ord(char) < 128) / len(text)
        number_ratio = sum(1 for char in text if char.isdigit()) / len(text)
        space_ratio = text.count(' ') / len(text)
        
        meaningless_chars = ['(cid:', '\\x', '\\u']
        meaningless_ratio = sum(text.count(char) for char in meaningless_chars) / len(text)
        
        # 품질 점수 계산
        quality_score = (korean_ratio * 0.4 + english_ratio * 0.3 + 
                        number_ratio * 0.1 + space_ratio * 0.1 - 
                        meaningless_ratio * 0.5)
        
        return max(0, min(1, quality_score))
    
    def _extract_with_pdfplumber_enhanced(self, pdf_path: Path, metadata: Dict) -> str:
        """pdfplumber 향상된 추출"""
        text_parts = []
        
        with pdfplumber.open(pdf_path) as pdf:
            metadata['page_count'] = len(pdf.pages)
            
            for page_num, page in enumerate(pdf.pages):
                # 다양한 추출 방법 시도
                methods = [
                    ('기본 추출', lambda p: p.extract_text()),
                    ('단어 기반', lambda p: p.extract_text(x_tolerance=3, y_tolerance=3)),
                    ('문자 기반', lambda p: ' '.join([char.get('text', '') for char in p.chars if char.get('text')])),
                ]
                
                page_text = ""
                for method_name, extract_func in methods:
                    try:
                        text = extract_func(page)
                        if text and self._get_text_quality_score(text) > 0.2:
                            page_text = text
                            break
                    except:
                        continue
                
                if page_text:
                    text_parts.append(f"=== 페이지 {page_num + 1} ===\\n{page_text}")
        
        return '\\n\\n'.join(text_parts)
    
    def _extract_with_pdfplumber_basic(self, pdf_path: Path, metadata: Dict) -> str:
        """pdfplumber 기본 추출"""
        text_parts = []
        
        with pdfplumber.open(pdf_path) as pdf:
            metadata['page_count'] = len(pdf.pages)
            
            for page_num, page in enumerate(pdf.pages):
                try:
                    text = page.extract_text()
                    if text:
                        text_parts.append(f"=== 페이지 {page_num + 1} ===\\n{text}")
                except:
                    continue
        
        return '\\n\\n'.join(text_parts)
    
    def _extract_with_pypdf2(self, pdf_path: Path, metadata: Dict) -> str:
        """PyPDF2를 사용한 텍스트 추출"""
        text_parts = []
        
        with open(pdf_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            metadata['page_count'] = len(pdf_reader.pages)
            
            for page_num, page in enumerate(pdf_reader.pages):
                try:
                    text = page.extract_text()
                    if text:
                        text_parts.append(f"=== 페이지 {page_num + 1} ===\\n{text}")
                except:
                    continue
        
        return '\\n\\n'.join(text_parts)
    
    def process_directory(self, directory: Path) -> List[Dict]:
        """
        디렉토리의 모든 PDF 파일 처리
        """
        pdf_files = list(directory.glob("*.pdf"))
        print(f"\n📁 {directory.name}에서 {len(pdf_files)}개 PDF 파일 발견")
        
        results = []
        
        for pdf_path in pdf_files:
            text, metadata = self.extract_text_from_pdf(pdf_path)
            
            if text and len(text.strip()) > 200:
                results.append({
                    'text': text,
                    'metadata': metadata
                })
                print(f"✅ {pdf_path.name} 처리 완료")
            else:
                print(f"❌ {pdf_path.name} 처리 실패")
        
        print(f"\n📊 PDF 추출 완료: 성공 {self.extracted_count}개, 실패 {self.failed_count}개")
        return results

### 3.3 OCR 텍스트 추출기

In [ ]:
class OCRTextExtractor:
    """
    OCR을 이용한 PDF 텍스트 추출 클래스
    PDF가 이미지나 스캔본인 경우에 사용
    """
    
    def __init__(self):
        self.extracted_count = 0
        self.failed_count = 0
        self.ocr_available = self._check_ocr_availability()
    
    def _check_ocr_availability(self) -> bool:
        """OCR 사용 가능 여부 확인"""
        try:
            import pytesseract
            import PIL
            from pdf2image import convert_from_path
            print("✅ OCR 라이브러리 사용 가능")
            return True
        except ImportError as e:
            print(f"⚠️  OCR 라이브러리 누락: {e}")
            print("설치 명령: pip install pytesseract pdf2image")
            print("Tesseract: conda install -c conda-forge tesseract")
            return False
    
    def extract_text_from_pdf_ocr(self, pdf_path: Path, max_pages: int = 10) -> Tuple[str, Dict]:
        """
        OCR을 사용하여 PDF에서 텍스트 추출
        """
        metadata = {
            'filename': pdf_path.name,
            'path': str(pdf_path),
            'size': pdf_path.stat().st_size,
            'extracted_at': datetime.now().isoformat(),
            'method': 'OCR',
            'page_count': 0,
            'max_pages_processed': max_pages
        }
        
        if not self.ocr_available:
            print(f"❌ OCR 라이브러리가 설치되지 않아 {pdf_path.name} 처리 불가")
            self.failed_count += 1
            return "", metadata
        
        print(f"\n🔍 {pdf_path.name} OCR 텍스트 추출 중...")
        
        try:
            from pdf2image import convert_from_path
            import pytesseract
            
            # PDF를 이미지로 변환 (메모리 절약을 위해 제한)
            print(f"   📄 PDF를 이미지로 변환 중... (최대 {max_pages}페이지)")
            images = convert_from_path(pdf_path, first_page=1, last_page=max_pages)
            metadata['page_count'] = len(images)
            
            text_parts = []
            successful_pages = 0
            
            for page_num, image in enumerate(images, 1):
                print(f"   🔍 페이지 {page_num} OCR 처리 중...")
                
                # 다양한 언어 설정 시도 (한국어 우선)
                ocr_configs = [
                    ('한국어', '--oem 3 --psm 6 -l kor'),
                    ('한국어+영어', '--oem 3 --psm 6 -l kor+eng'),
                    ('영어', '--oem 3 --psm 6 -l eng')
                ]
                
                page_text = ""
                for lang_name, config in ocr_configs:
                    try:
                        text = pytesseract.image_to_string(image, config=config)
                        
                        if text and len(text.strip()) > 50:  # 최소 50자
                            # 한글 문자 개수 확인
                            korean_count = sum(1 for char in text if '\uac00' <= char <= '\ud7a3')
                            english_count = sum(1 for char in text if char.isalpha() and ord(char) < 128)
                            
                            page_text = text
                            print(f"     ✓ {lang_name} OCR 성공: {len(text)}자 (한글: {korean_count}, 영문: {english_count})")
                            break
                        
                    except Exception as e:
                        if 'kor' in lang_name and 'Error opening data file' in str(e):
                            print(f"     ⚠ 한국어 언어팩 없음")
                            continue
                        else:
                            print(f"     ❌ {lang_name} OCR 실패: {str(e)[:50]}...")
                            continue
                
                if page_text:
                    text_parts.append(f"=== 페이지 {page_num} (OCR) ===\\n{page_text}")
                    successful_pages += 1
                else:
                    print(f"     ❌ 페이지 {page_num} OCR 실패")
            
            # 결과 처리
            result_text = '\\n\\n'.join(text_parts)
            
            if successful_pages > 0 and len(result_text.strip()) > 200:
                print(f"   ✅ OCR 성공: {successful_pages}/{len(images)} 페이지 처리, {len(result_text):,}자 추출")
                
                # OCR 결과를 별도 파일로도 저장 (디버깅용)
                ocr_debug_file = OUTPUT_DIR / f"ocr_result_{pdf_path.stem}.txt"
                with open(ocr_debug_file, 'w', encoding='utf-8') as f:
                    f.write(result_text)
                print(f"   💾 OCR 결과 저장: {ocr_debug_file.name}")
                
                metadata['successful_pages'] = successful_pages
                self.extracted_count += 1
                return result_text, metadata
            else:
                print(f"   ❌ OCR 추출 실패: 성공 페이지 {successful_pages}개, 텍스트 {len(result_text)}자")
                self.failed_count += 1
                metadata['error'] = f"OCR 추출 부족: {successful_pages}페이지, {len(result_text)}자"
                return "", metadata
            
        except Exception as e:
            print(f"   ❌ OCR 처리 실패: {str(e)}")
            self.failed_count += 1
            metadata['error'] = str(e)
            return "", metadata
    
    def process_directory(self, directory: Path, max_pages: int = 10) -> List[Dict]:
        """
        디렉토리의 모든 PDF 파일을 OCR로 처리
        """
        if not self.ocr_available:
            print("❌ OCR 라이브러리가 설치되지 않아 처리할 수 없습니다.")
            return []
        
        pdf_files = list(directory.glob("*.pdf"))
        print(f"\n📁 {directory.name}에서 {len(pdf_files)}개 PDF 파일을 OCR로 처리")
        
        results = []
        
        for pdf_path in pdf_files:
            text, metadata = self.extract_text_from_pdf_ocr(pdf_path, max_pages)
            
            if text and len(text.strip()) > 200:
                results.append({
                    'text': text,
                    'metadata': metadata
                })
                print(f"✅ {pdf_path.name} OCR 완료")
            else:
                print(f"❌ {pdf_path.name} OCR 실패")
        
        print(f"\n📊 OCR 처리 완료: 성공 {self.extracted_count}개, 실패 {self.failed_count}개")
        return results

## 4. LLM 기반 농업 콘텐츠 검증 클래스

In [ ]:
class AgriculturalContentValidator:
    """
    LLM을 사용하여 텍스트가 농업 관련 내용인지 검증하는 클래스
    """
    
    def __init__(self, client: OpenAI, model: str = "gpt-4o-mini", debug: bool = True):
        self.client = client
        self.model = model
        self.debug = debug
        self.validated_count = 0
        self.rejected_count = 0
    
    def validate_agricultural_content(self, text: str, sample_size: int = 1500) -> Tuple[bool, Dict]:
        """
        텍스트가 농업 관련 내용인지 LLM으로 검증
        """
        # 텍스트 샘플 추출
        text_sample = text[:sample_size] if len(text) > sample_size else text
        
        if self.debug:
            print(f"\n{'='*50}")
            print(f"📄 텍스트 검증 시작")
            print(f"📊 텍스트 길이: {len(text):,}자")
            print(f"{'='*50}")
        
        # 구체적이고 명확한 프롬프트
        prompt = f"""다음 텍스트가 농업과 관련된 내용인지 판단해주세요.

농업 관련 내용 기준:
- 작물 재배, 품종, 씨앗, 수확, 농업 기술
- 축산, 가축, 사료, 동물 관리
- 농업 정책, 농촌 개발, 농업 경제
- 농업 기계, 농업 도구, 농업 시설
- 병충해, 농약, 비료, 토양
- 식품 생산, 농산물 가공
- 농업 연구, 농업 교육

텍스트 내용:
{text_sample}

반드시 다음 형식으로 답변하세요:
1. 농업 관련 여부: 예/아니오
2. 발견된 농업 키워드: (예: 작물, 재배, 수확 등)
3. 판단 근거: (왜 농업 관련이거나 아닌지 구체적 설명)
4. 신뢰도: 높음/중간/낮음"""
        
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": "당신은 농업 분야 전문가입니다. 텍스트가 농업과 관련이 있는지 정확하게 판단하세요. 농업과 조금이라도 관련이 있다면 '예'로 답변하세요."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.1,
                max_tokens=400
            )
            
            result_text = response.choices[0].message.content
            
            if self.debug:
                print(f"\n🤖 LLM 검증 결과:")
                print("-" * 50)
                print(result_text[:200])
                print("-" * 50)
            
            # 결과 파싱
            first_line = result_text.split('\n')[0]
            is_agricultural = ("예" in first_line) or ("농업 관련 여부: 예" in result_text)
            
            validation_result = {
                'is_agricultural': is_agricultural,
                'llm_response': result_text,
                'text_length': len(text),
                'sample_length': len(text_sample),
                'validated_at': datetime.now().isoformat()
            }
            
            if is_agricultural:
                self.validated_count += 1
                status = "✅ 농업 관련 콘텐츠로 검증됨"
            else:
                self.rejected_count += 1
                status = "❌ 농업 관련이 아닌 것으로 판단됨"
                
            if self.debug:
                print(f"\n{status}")
            
            return is_agricultural, validation_result
            
        except Exception as e:
            logger.error(f"LLM 검증 실패: {e}")
            # 에러 시 기본적으로 통과시킴 
            return True, {
                'is_agricultural': True,
                'error': str(e),
                'note': '검증 실패로 인해 기본 통과 처리'
            }
    
    def get_statistics(self) -> Dict:
        """검증 통계 반환"""
        total = self.validated_count + self.rejected_count
        return {
            'total_processed': total,
            'validated': self.validated_count,
            'rejected': self.rejected_count,
            'validation_rate': self.validated_count / total if total > 0 else 0
        }

In [ ]:
def run_agricultural_pipeline(source_type: str = "all", include_ocr: bool = False, 
                             debug_mode: bool = True, num_qa_pairs: int = 2):
    """
    농업 텍스트 추출 및 QA 생성 파이프라인 실행
    
    Args:
        source_type: 소스 타입 ("arxiv", "rda", "all", "text_units")
        include_ocr: OCR 사용 여부 (기본값: False)
        debug_mode: 디버깅 모드 활성화 여부
        num_qa_pairs: 생성할 QA 쌍의 개수 (기본값: 2)
    """
    print("="*80)
    print("🚀 농업 텍스트 데이터 QA 파이프라인 시작")
    print(f"📝 소스: {source_type.upper()}")
    print(f"🔍 OCR 사용: {'예' if include_ocr else '아니오'}")
    print(f"🔍 디버깅 모드: {'ON' if debug_mode else 'OFF'}")
    print(f"🎯 QA 쌍 개수: {num_qa_pairs}개")
    print("="*80)
    
    # 1단계: 텍스트 추출
    extracted_results = []
    
    # text_units.parquet 파일에서 데이터 로드
    if source_type == "text_units":
        print("\n[1/4] 📊 text_units.parquet에서 텍스트 로드 중...")
        import pandas as pd
        
        text_units_path = Path("<PROJECT_ROOT>/kg_gen/snu_kg_output/graphrag_workspace/output_after_kggen/text_units.parquet")
        
        try:
            df = pd.read_parquet(text_units_path)
            print(f"✅ {len(df)}개 텍스트 유닛 로드 완료")
            
            # DataFrame을 extracted_results 형식으로 변환
            for idx, row in df.iterrows():
                extracted_results.append({
                    'text': row['text'],
                    'metadata': {
                        'source': 'text_units.parquet',
                        'id': row['id'],
                        'human_readable_id': row['human_readable_id'],
                        'n_tokens': row['n_tokens'],
                        'method': 'parquet',
                        'index': idx
                    }
                })
            
            print(f"📊 평균 텍스트 길이: {df['text'].str.len().mean():.1f}자")
            print(f"📊 총 텍스트 유닛: {len(extracted_results)}개")
            
        except Exception as e:
            print(f"❌ text_units.parquet 로드 실패: {e}")
            return None
    
    # arXiv PDF 처리
    elif source_type in ["arxiv", "all"]:
        print("\n[1/4] 📄 arXiv PDF에서 텍스트 추출 중...")
        
        if include_ocr:
            # OCR 사용
            ocr_extractor = OCRTextExtractor()
            results = ocr_extractor.process_directory(ARXIV_DIR, max_pages=10)
        else:
            # PDF 직접 추출
            pdf_extractor = PDFTextExtractor()
            results = pdf_extractor.process_directory(ARXIV_DIR)
        
        extracted_results.extend(results)
        print(f"arXiv: {len(results)}개 PDF 처리됨")
    
    # RDA 웹 스크래핑
    if source_type in ["rda", "all"]:
        print("\n[1/4] 🌐 RDA URL에서 웹 스크래핑 중...")
        web_extractor = WebScrapingExtractor()
        
        # RDA URL 파일 읽기
        if RDA_URL_FILE.exists():
            urls = web_extractor.load_urls_from_file(RDA_URL_FILE)
            if urls:
                print(f"📋 {len(urls)}개 RDA URL 발견")
                results = web_extractor.extract_from_url_list(urls, delay=1.5)
                extracted_results.extend(results)
            else:
                print(f"❌ RDA URL 파일이 비어있습니다.")
        else:
            print(f"❌ RDA URL 파일이 없습니다: {RDA_URL_FILE}")
    
    print(f"\n📊 1단계 완료: {len(extracted_results)}개 텍스트 추출됨")
    
    if len(extracted_results) == 0:
        print("⚠️  추출된 텍스트가 없습니다.")
        return None
    
    # 첫 번째 추출 결과 미리보기
    if debug_mode and extracted_results:
        print(f"\n🔍 첫 번째 추출 결과 미리보기:")
        first_result = extracted_results[0]
        print(f"   방식: {first_result['metadata'].get('method', 'unknown')}")
        if 'filename' in first_result['metadata']:
            print(f"   파일: {first_result['metadata']['filename']}")
        elif 'url' in first_result['metadata']:
            print(f"   URL: {first_result['metadata']['url'][:80]}...")
        elif 'id' in first_result['metadata']:
            print(f"   ID: {first_result['metadata']['id'][:40]}...")
        print(f"   텍스트 길이: {len(first_result['text']):,}자")
        print("\n📄 텍스트 일부:")
        print("-" * 60)
        print(first_result['text'][:500])
        if len(first_result['text']) > 500:
            print("...(중략)...")
        print("-" * 60)
    
    # 2단계: LLM 농업 콘텐츠 검증 (text_units는 이미 농업 관련이므로 생략 가능)
    if source_type != "text_units":
        print(f"\n[2/4] 🤖 LLM 농업 콘텐츠 검증 중...")
        validator = AgriculturalContentValidator(client, debug=debug_mode)
        validated_texts = []
        
        for i, result in enumerate(extracted_results):
            text = result['text']
            metadata = result['metadata']
            
            source_info = metadata.get('filename', metadata.get('url', f'항목{i+1}'))
            print(f"\n📋 {i+1}/{len(extracted_results)} 검증 중: {source_info[:80]}...")
            
            is_agricultural, validation_result = validator.validate_agricultural_content(text)
            
            if is_agricultural:
                validated_texts.append({
                    'text': text,
                    'metadata': metadata,
                    'validation': validation_result
                })
                print(f"✅ 농업 관련 콘텐츠로 승인됨")
            else:
                print(f"❌ 농업 관련 아님으로 제외됨")
            
            # API 호출 제한
            time.sleep(1)
        
        validation_stats = validator.get_statistics()
        print(f"\n📊 2단계 완료:")
        print(f"   - 총 처리: {validation_stats['total_processed']}개")
        print(f"   - 통과: {validation_stats['validated']}개")
        print(f"   - 제외: {validation_stats['rejected']}개") 
        print(f"   - 통과율: {validation_stats['validation_rate']:.1%}")
    else:
        # text_units는 검증 단계 생략
        print(f"\n[2/4] ✅ text_units는 이미 농업 관련 텍스트이므로 검증 생략")
        validated_texts = [{'text': r['text'], 'metadata': r['metadata'], 'validation': {'is_agricultural': True}} for r in extracted_results]
    
    if len(validated_texts) == 0:
        print("⚠️  농업 관련으로 검증된 텍스트가 없습니다.")
        return None
    
    # 3단계: 검증된 텍스트 저장
    print(f"\n[3/4] 💾 검증된 텍스트 저장 중...")
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    validated_file = OUTPUT_DIR / f"validated_texts_{source_type}_{timestamp}.json"
    
    with open(validated_file, 'w', encoding='utf-8') as f:
        json.dump(validated_texts, f, ensure_ascii=False, indent=2)
    
    print(f"✅ 검증된 텍스트 저장 완료: {validated_file.name}")
    
    # 4단계: QA 데이터셋 생성
    print(f"\n[4/4] 🎯 QA 데이터셋 생성 중... (텍스트당 {num_qa_pairs}개 QA)")
    all_qa_data = []
    
    for validated_item in tqdm(validated_texts, desc="QA 생성"):
        text = validated_item['text']
        metadata = validated_item['metadata']
        
        if source_type == "text_units":
            source_info = f"text_unit_{metadata.get('index', 'unknown')}"
        else:
            source_info = metadata.get('filename', metadata.get('url', '알 수 없는 소스'))
        
        # text_units는 이미 적절한 크기이므로 청킹 필요 없음
        if source_type == "text_units":
            # 300 토큰 이상인 텍스트만 처리
            if len(text) >= 300:  # 대략적으로 300자 이상
                qa_text = generate_qa_from_text(text, num_qa_pairs)
                if qa_text:
                    qa_dict = parse_qa_response(qa_text, num_qa_pairs)
                    if qa_dict:
                        qa_entry = {
                            'source': source_info,
                            'source_type': 'text_units',
                            'text_unit_id': metadata.get('id', ''),
                            'source_text': text,
                            'generated_at': datetime.now().isoformat(),
                            'num_qa_pairs': num_qa_pairs
                        }
                        
                        # 동적으로 QA 쌍 추가
                        for qa_key, qa_value in qa_dict.items():
                            qa_entry[qa_key] = qa_value
                        
                        all_qa_data.append(qa_entry)
            else:
                print(f"⏩ 텍스트가 너무 짧음 ({len(text)}자), 건너뜀")
        else:
            # 기존 소스는 청킹 필요
            chunks = chunk_text(text)
            print(f"📄 {source_info[:50]}...: {len(chunks)}개 청크 생성")
            
            for chunk_idx, chunk in enumerate(chunks):
                # QA 생성 (설정된 개수만큼)
                qa_text = generate_qa_from_text(chunk, num_qa_pairs)
                if qa_text:
                    qa_dict = parse_qa_response(qa_text, num_qa_pairs)
                    if qa_dict:
                        qa_entry = {
                            'source': source_info,
                            'source_type': 'arxiv' if 'filename' in metadata else 'rda',
                            'chunk_index': chunk_idx,
                            'source_text': chunk,
                            'generated_at': datetime.now().isoformat(),
                            'num_qa_pairs': num_qa_pairs
                        }
                        
                        # 동적으로 QA 쌍 추가
                        for qa_key, qa_value in qa_dict.items():
                            qa_entry[qa_key] = qa_value
                        
                        all_qa_data.append(qa_entry)
        
        # API 호출 제한
        time.sleep(1)
        
        # 100개마다 중간 저장
        if len(all_qa_data) > 0 and len(all_qa_data) % 100 == 0:
            batch_file = QA_DIR / f"qa_batch_{source_type}_{len(all_qa_data)//100}_{timestamp}.json"
            with open(batch_file, 'w', encoding='utf-8') as f:
                json.dump(all_qa_data[-100:], f, ensure_ascii=False, indent=2)
            print(f"💾 중간 저장: {batch_file.name}")
    
    # 최종 QA 데이터셋 저장
    final_qa_file = QA_DIR / f"qa_dataset_{source_type}_final_{timestamp}.json"
    with open(final_qa_file, 'w', encoding='utf-8') as f:
        json.dump(all_qa_data, f, ensure_ascii=False, indent=2)
    
    print(f"\n🎉 파이프라인 완료!")
    print(f"📊 최종 결과:")
    print(f"   - 소스 타입: {source_type.upper()}")
    print(f"   - 처리된 소스: {len(extracted_results)}개")
    print(f"   - 검증된 텍스트: {len(validated_texts)}개") 
    print(f"   - 생성된 QA 쌍: {len(all_qa_data)}개")
    print(f"   - 청크당 QA 개수: {num_qa_pairs}개")
    print(f"📁 파일 저장:")
    print(f"   - 검증된 텍스트: {validated_file.name}")
    print(f"   - QA 데이터셋: {final_qa_file.name}")
    
    return {
        'source_type': source_type,
        'extracted_sources': len(extracted_results),
        'validated_texts': len(validated_texts),
        'qa_pairs': len(all_qa_data),
        'num_qa_pairs': num_qa_pairs,
        'files': {
            'validated_texts': str(validated_file),
            'qa_dataset': str(final_qa_file)
        }
    }

## 7. 파이프라인 실행

In [ ]:
def generate_qa_from_text(text_line, num_qa_pairs=1):
    """
    텍스트에서 QA 쌍을 생성하는 함수 (QA 개수 설정 가능)
    
    Args:
        text_line: 텍스트 내용
        num_qa_pairs: 생성할 QA 쌍의 개수 (기본값: 2)
    """
    # QA 개수에 따른 프롬프트 생성
    qa_format_parts = []
    for i in range(1, num_qa_pairs + 1):
        qa_format_parts.append(f"Q{i}:")
        qa_format_parts.append(f"A{i}:")
    
    qa_format = "\n".join(qa_format_parts)
    
    if num_qa_pairs == 1:
        instruction = "Generate one question and corresponding answer"
    else:
        instruction = f"Generate {num_qa_pairs} questions and corresponding answers"
    
    messages = [
        {"role": "system", "content": "You are a helpful assistant that generates question and answer pairs using Korean based on given text."},
        {"role": "user", "content": f"""{instruction} based on the following text:

Text: {text_line}

Output must be in the following format:
{qa_format}
"""}
    ]
    
    try:
        # QA 개수에 따라 max_tokens 조정
        max_tokens = min(150 + (num_qa_pairs * 50), 500)
        
        response = client.chat.completions.create(
            model="gpt-4o-mini-2024-07-18",
            messages=messages,
            max_tokens=max_tokens,
            temperature=0.7
        )
        
        return response.choices[0].message.content
        
    except Exception as e:
        logger.error(f"QA 생성 실패: {e}")
        return None


def parse_qa_response(generated_text: str, num_qa_pairs=2) -> Dict[str, str]:
    """
    생성된 QA 텍스트를 파싱 (QA 개수에 따라 동적 파싱)
    
    Args:
        generated_text: LLM이 생성한 QA 텍스트
        num_qa_pairs: 예상되는 QA 쌍의 개수
    """
    if not generated_text:
        return None
    
    try:
        result = {}
        
        # Q1부터 시작해서 순차적으로 파싱
        for i in range(1, num_qa_pairs + 1):
            q_pattern = f"Q{i}:"
            a_pattern = f"A{i}:"
            
            # 현재 Q 찾기
            q_start = generated_text.find(q_pattern)
            if q_start == -1:
                break
                
            # 다음 Q 또는 텍스트 끝 찾기
            next_q_start = generated_text.find(f"Q{i+1}:", q_start + len(q_pattern))
            if next_q_start == -1:
                qa_text = generated_text[q_start:].strip()
            else:
                qa_text = generated_text[q_start:next_q_start].strip()
            
            result[f"QA{i}"] = qa_text
        
        # 최소 1개의 QA는 있어야 함
        if len(result) == 0:
            result["QA1"] = generated_text.strip()
        
        return result
    
    except Exception as e:
        logger.error(f"QA 파싱 실패: {e}")
        return {"QA1": generated_text.strip()} if generated_text else None


def chunk_text(text: str, chunk_size: int = 2390) -> List[str]:
    """텍스트를 청크로 분할 (AgriLLM.ipynb 방식)"""
    chunks = []
    
    for i in range(0, len(text), chunk_size):
        chunk = text[i:i + chunk_size]
        chunks.append(chunk)
    
    # 500자 이상인 청크만 필터링
    filtered_chunks = [chunk for chunk in chunks if len(chunk) >= 500]
    
    return filtered_chunks

## 5. QA 생성 함수

In [ ]:
# 파이프라인 실행 예시
# 원하는 옵션을 선택하여 실행하세요

# 예시 1: text_units.parquet 파일 사용 (기본 설정, QA 2개씩)
results = run_agricultural_pipeline(source_type="text_units")

# 예시 2: text_units에서 QA 3개씩 생성
# results = run_agricultural_pipeline(source_type="text_units", num_qa_pairs=3)

# 예시 3: text_units에서 QA 5개씩 생성
# results = run_agricultural_pipeline(source_type="text_units", num_qa_pairs=5)

# 예시 4: 기존 방식 - 모든 소스 사용
# results = run_agricultural_pipeline(source_type="all", num_qa_pairs=2)

# 예시 5: 기존 방식 - RDA만 사용
# results = run_agricultural_pipeline(source_type="rda", num_qa_pairs=2)

In [ ]:
# 🚀 파이프라인 실행 가이드

print("="*80)
print("📁 파일 준비 상태 확인:")
print("="*80)

# text_units.parquet 파일 확인
text_units_path = Path("<PROJECT_ROOT>/kg_gen/snu_kg_output/graphrag_workspace/output_after_kggen/text_units.parquet")
if text_units_path.exists():
    import pandas as pd
    df = pd.read_parquet(text_units_path)
    print(f"📊 text_units.parquet 파일:")
    print(f"   - 텍스트 유닛 수: {len(df)}개")
    print(f"   - 평균 텍스트 길이: {df['text'].str.len().mean():.1f}자")
    print(f"   - 최소/최대 길이: {df['text'].str.len().min()}자 / {df['text'].str.len().max()}자")
    print(f"   - 300자 이상 텍스트: {(df['text'].str.len() >= 300).sum()}개")
else:
    print("❌ text_units.parquet 파일이 없습니다.")

# 기존 소스 파일 확인
print("\n📄 기존 소스 파일 상태:")

# arXiv PDF 파일 확인
arxiv_files = list(ARXIV_DIR.glob("*.pdf"))
print(f"\n📄 arXiv PDF 파일: {len(arxiv_files)}개")
for pdf in arxiv_files[:3]:  # 처음 3개만 표시
    print(f"   - {pdf.name}")
if len(arxiv_files) > 3:
    print(f"   ... 외 {len(arxiv_files) - 3}개")

# RDA URL 파일 확인  
print(f"\n🌐 RDA URL 파일: {RDA_URL_FILE.name}")
if RDA_URL_FILE.exists():
    try:
        with open(RDA_URL_FILE, 'r', encoding='utf-8') as f:
            urls = [line.strip() for line in f if line.strip() and not line.startswith('#')]
            print(f"   - {len(urls)}개 URL 저장됨")
            for url in urls[:3]:  # 처음 3개만 표시
                print(f"     • {url[:80]}...")
            if len(urls) > 3:
                print(f"     ... 외 {len(urls) - 3}개")
    except:
        print("   - 파일 읽기 실패")
else:
    print("   - 파일 없음 (생성됨)")

print(f"\n💡 사용 가능한 소스:")
if text_units_path.exists():
    print("   ✅ 'text_units' - text_units.parquet 파일 (권장)")
else:
    print("   ❌ 'text_units' - parquet 파일이 없음")

if arxiv_files:
    print("   ✅ 'arxiv' - arXiv 논문 PDF")
else:
    print("   ❌ 'arxiv' - PDF 파일이 없음")

if RDA_URL_FILE.exists() and len(urls) > 0:
    print("   ✅ 'rda' - RDA 웹 스크래핑")
else:
    print("   ❌ 'rda' - URL이 없음")

if arxiv_files or (RDA_URL_FILE.exists() and len(urls) > 0):
    print("   ✅ 'all' - 모든 소스 처리")

print("\n" + "="*80)
print("📝 실행 예시:")
print("="*80)
print("# text_units.parquet 사용 (권장)")
print("results = run_agricultural_pipeline(source_type='text_units')")
print("\n# text_units에서 QA 3개씩 생성")
print("results = run_agricultural_pipeline(source_type='text_units', num_qa_pairs=3)")
print("\n# 기존 방식 - arXiv PDF만 처리")
print("results = run_agricultural_pipeline(source_type='arxiv')")
print("\n# 기존 방식 - RDA URL만 처리")  
print("results = run_agricultural_pipeline(source_type='rda')")
print("\n# 기존 방식 - 모든 소스 처리")
print("results = run_agricultural_pipeline(source_type='all')")
print("\n# OCR 사용 (폰트 문제가 있는 PDF인 경우)")
print("results = run_agricultural_pipeline(source_type='arxiv', include_ocr=True)")
print("="*80)